# Experiment: Generate all coursework depth maps with DA3 FP16 on Colab

Generate **all 5,686 frames** with the pinned DA3 Metric Large model: KITTI 06 (1,101), TUM fr3 (2,585), and Replica room0 (2,000). Inference uses **CUDA FP16 at processing resolution 504**. Each frame produces metric float32 NPY, scaled uint16 PNG and a checksummed manifest entry.

1. Open this notebook in [Colab](https://colab.research.google.com/).
2. Choose **Runtime > Change runtime type > GPU**, with **Runtime version 2026.07** (Python 3.12).
3. Use Colab's **Files** sidebar to upload **da3_full_inputs.zip** (about 3.02 GB) to `/content`, then run the cells in order.
4. Download each dataset ZIP manually through the **Files** sidebar, one at a time, and save the small metadata ZIP. Confirm downloads finish before ending the runtime.

This uses an available free Colab GPU; allocation and session length are not guaranteed. The notebook stops if CUDA or sufficient disk is unavailable and never switches to CPU or purchases compute. It does not mount Google Drive. The output cache is committed after every frame, but local VM storage can disappear after a runtime reset. Download completed datasets promptly; the rescue cell can export a partial dataset before you terminate a surviving session.

Input images are uploaded once. The DA3 checkpoint is downloaded directly from its pinned official location (1.337 GB plus an 847-byte configuration). Existing smoke-test environment/source/weights in the same Colab VM are reused when compatible. This notebook performs depth generation; it does not run SLAM or score depth against GT.

References: [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html), [free resource limits](https://research.google.com/colaboratory/faq.html), [official DA3](https://github.com/ByteDance-Seed/Depth-Anything-3), [DA3 Metric Large](https://huggingface.co/depth-anything/DA3METRIC-LARGE).

## 1. Check CUDA, Python and available disk

A conservative **45 GiB free** budget covers the large input ZIP and extracted images, weights, environments, NPY/PNG caches and all three result ZIPs. The notebook does not delete other work to make space. Probe processes leave the notebook kernel's numerical packages untouched.

In [ ]:
import datetime
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Select Runtime > Change runtime type > Runtime version 2026.07, then reconnect.')
probe_code = """import json, torch
available = torch.cuda.is_available()
print(json.dumps({'cuda_available': available, 'torch': torch.__version__, 'cuda_version': torch.version.cuda,
                  'gpu': torch.cuda.get_device_name(0) if available else None,
                  'gpu_memory_gib': torch.cuda.get_device_properties(0).total_memory / 2**30 if available else None}))
"""
gpu_info = json.loads(subprocess.check_output([sys.executable, '-c', probe_code], text=True))
if not gpu_info['cuda_available']:
    raise RuntimeError('No CUDA GPU allocation. Stop here and try a GPU runtime later; do not run the cases on CPU.')
if shutil.disk_usage('/content').free < 45 * 1024**3:
    raise RuntimeError('At least 45 GiB of free temporary disk is needed for inputs, full caches and result ZIPs.')
run_env = {**os.environ, 'PYTHONDONTWRITEBYTECODE': '1', 'NVIDIA_TF32_OVERRIDE': '0',
           'MPLBACKEND': 'Agg', 'TOKENIZERS_PARALLELISM': 'false'}
nvidia_info = subprocess.check_output(['nvidia-smi'], text=True)
print(json.dumps(gpu_info, indent=2))
print(nvidia_info)

## 2. Upload through the Files sidebar, then extract from disk

Open the folder icon on Colab's left sidebar, choose **Upload**, and select **da3_full_inputs.zip**. Wait for the approximately 3.02 GB transfer to finish. Its default path is `/content/da3_full_inputs.zip`; change `INPUT_ARCHIVE` below only if you placed it elsewhere.

This reads the ZIP by filename and hashes it in small chunks. It does not load a multi-gigabyte upload dictionary into the notebook kernel. Do not use the nine-frame smoke ZIP here. The full bundle contains RGB inputs, timestamps, calibration and scripts; GT depth and weights are excluded.

In [ ]:
from google.colab import files
import re
import stat
import zipfile

INPUT_ARCHIVE = '/content/da3_full_inputs.zip'
EXPECTED_INPUT_SHA256 = '2f168fad787de3fb03f7f9b0312b267d8a2aff0f40f82e7a4869868101f7e9f5'
input_zip = Path(INPUT_ARCHIVE).expanduser().resolve()
if not input_zip.is_file():
    raise RuntimeError('Upload da3_full_inputs.zip using the Colab Files sidebar, wait for completion, then rerun this cell.')
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INPUT_SHA256):
    raise RuntimeError('This notebook needs its prepared input checksum before use')
digest = hashlib.sha256()
with input_zip.open('rb') as handle:
    for block in iter(lambda: handle.read(8 * 1024**2), b''):
        digest.update(block)
input_sha256 = digest.hexdigest()
if input_sha256 != EXPECTED_INPUT_SHA256:
    raise ValueError('Input ZIP SHA256 mismatch; use the matching full input bundle')
workspace = Path('/content/cw1_da3_full_run')
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(input_zip) as archive:
    entries = archive.infolist()
    if len(entries) > 6000 or sum(entry.file_size for entry in entries) > 6 * 1024**3:
        raise ValueError('Unexpected full input archive size or file count')
    if len({entry.filename for entry in entries}) != len(entries):
        raise ValueError('Duplicate archive members are not accepted')
    for entry in entries:
        member = Path(entry.filename)
        destination = (workspace / member).resolve()
        if (member.is_absolute() or '..' in member.parts or not member.parts
                or member.parts[0] != 'cw1_da3_full'
                or not destination.is_relative_to(workspace.resolve())
                or stat.S_ISLNK(entry.external_attr >> 16)):
            raise ValueError(f'Unexpected archive member: {entry.filename}')
    archive.extractall(workspace)
project = workspace / 'cw1_da3_full'
input_manifest_path = project / 'full_input_manifest.json'
input_manifest = json.loads(input_manifest_path.read_text())
expected_counts = {'kitti06': 1101, 'tum_fr3': 2585, 'replica_room0': 2000}
if input_manifest.get('schema_version') != 1 or input_manifest.get('total_frames') != 5686:
    raise ValueError('Expected the complete 5,686-frame input manifest')
if set(input_manifest.get('sequences',{})) != set(expected_counts):
    raise ValueError('The full input manifest must contain exactly the three coursework sequences')
for sequence, count in expected_counts.items():
    record = input_manifest['sequences'][sequence]
    if record['count'] != count or len(record['frames']) != count:
        raise ValueError(f'Incorrect full frame count for {sequence}')
for filename in ('predict_da3.py','transfer_da3_cache.py','da3_adapter.py','common.py'):
    if not (project / 'scripts/part2' / filename).is_file():
        raise FileNotFoundError(f'Missing full-run helper {filename}')
run_env['CW1_PYSLAM_ROOT'] = str(project / 'calibration')
pred_root = project / 'depth_maps/da3_metric_large_fp16'
results_root = project / 'results/da3_full'
results_root.mkdir(parents=True, exist_ok=True)
(results_root / 'full_input_manifest.json').write_bytes(input_manifest_path.read_bytes())
(results_root / 'nvidia_smi.txt').write_text(nvidia_info)
(results_root / 'runtime.json').write_text(json.dumps({
    'started_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'gpu': gpu_info,
    'input_zip_sha256': input_sha256, 'input_manifest_sha256': hashlib.sha256(input_manifest_path.read_bytes()).hexdigest(),
    'precision': 'float16', 'process_res': 504, 'device': 'cuda', 'tf32_override': '0',
    'expected_counts': expected_counts,
}, indent=2))
print('Extracted full input bundle:', sum(expected_counts.values()), 'frames')

## 3. Reuse or prepare the isolated DA3 environment

This uses the same venv path as the DA3 smoke notebook. It inherits Colab's CUDA PyTorch/Torchvision and installs the small support packages only when the venv has not already been prepared. DA3 app/export dependencies, xformers and Open3D are not installed.

In [ ]:
env_root = Path('/content/cw1_da3_smoke_env')
python = env_root / 'bin/python'
new_environment = not python.is_file()
if new_environment:
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(env_root)], check=True)
dependency_probe = 'import numpy, cv2, scipy, pandas, yaml, matplotlib, timm, einops, omegaconf, addict, imageio, tqdm, safetensors; assert numpy.__version__ == "1.26.4"'
prepare_environment = new_environment or subprocess.run([str(python), '-c', dependency_probe],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0
if prepare_environment:
    subprocess.run([str(python), '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
                'numpy==1.26.4', 'opencv-python-headless==4.10.0.84', 'timm==1.0.30',
                'scipy', 'pandas', 'pyyaml', 'matplotlib', 'einops',
                'omegaconf', 'addict', 'imageio', 'tqdm', 'safetensors', 'huggingface-hub'], check=True)
env_gpu_info = json.loads(subprocess.check_output([str(python), '-c', probe_code], text=True, env=run_env))
if (not env_gpu_info['cuda_available'] or env_gpu_info['torch'] != gpu_info['torch']
        or env_gpu_info['cuda_version'] != gpu_info['cuda_version']):
    raise RuntimeError('CUDA/PyTorch changed unexpectedly; stop and inspect this environment')
subprocess.run([str(python), '-c',
                'import numpy, cv2, torch, torchvision; assert numpy.__version__ == "1.26.4"; '
                'print("NumPy", numpy.__version__, "OpenCV", cv2.__version__, "CUDA", torch.cuda.is_available())'],
               check=True, env=run_env)
freeze = subprocess.check_output([str(python), '-m', 'pip', 'freeze'], text=True)
(results_root / 'environment.txt').write_text(freeze)
(results_root / 'venv_gpu.json').write_text(json.dumps(env_gpu_info, indent=2))

## 4. Verify all frame identities and preview the full run before weights

This streams SHA256 checks over all 5,686 RGB files and verifies indices, timestamps and camera calibration against the full input manifest. It then runs the cache script's inventory mode, which does not import Torch or load the network.

In [ ]:
full_inventory_code = r"""
import hashlib, json, math, sys
from pathlib import Path
sys.path.insert(0,'scripts')
from part2.common import load_camera, load_frame_records
project=Path(sys.argv[1]).resolve()
manifest=json.loads((project/'full_input_manifest.json').read_text())
counts={'kitti06':1101,'tum_fr3':2585,'replica_room0':2000}
checked=0
for sequence,count in counts.items():
    item=manifest['sequences'][sequence]
    records=load_frame_records(sequence,data_root=project/'data')
    camera=load_camera(sequence,data_root=project/'data')
    assert len(records)==count==item['count']==len(item['frames'])
    for key in ('fx','fy','cx','cy','width','height'):
        assert math.isclose(float(getattr(camera,key)),float(item['camera'][key]),rel_tol=0,abs_tol=1e-7)
    previous=None
    for position,(record,row) in enumerate(zip(records,item['frames'])):
        assert record.index==row['index']==position
        assert math.isfinite(row['timestamp']) and math.isclose(record.timestamp,row['timestamp'],rel_tol=0,abs_tol=1e-6)
        assert previous is None or row['timestamp']>previous
        previous=row['timestamp']
        relative=Path(row['rgb'])
        path=(project/relative).resolve()
        assert not relative.is_absolute() and relative.parts[0]=='data' and '..' not in relative.parts
        assert path.is_relative_to(project) and path==record.rgb.resolve()
        digest=hashlib.sha256()
        with path.open('rb') as handle:
            for block in iter(lambda:handle.read(8*1024**2),b''):
                digest.update(block)
        assert digest.hexdigest()==row['rgb_sha256'],f'RGB checksum mismatch: {sequence}/{record.stem}'
        checked+=1
        if checked%1000==0: print(f'Checked {checked}/5686 RGB files',flush=True)
    print(f'{sequence}: {count} original indices, timestamps, RGB hashes and calibration verified',flush=True)
assert checked==manifest['total_frames']==5686
print('Full RGB input inventory passed: 5686/5686',flush=True)
"""
subprocess.run([str(python), '-c', full_inventory_code, str(project)], cwd=project, env=run_env, check=True)
source_root = Path('/content/cw1_da3_smoke_source')
weights_root = Path('/content/cw1_da3_smoke_weights')
checkpoint_dir = weights_root / 'da3metric-large'
subprocess.run([str(python), str(project / 'scripts/part2/predict_da3.py'), '--all',
                '--checkpoint', str(checkpoint_dir), '--source-root', str(source_root),
                '--data-root', str(project / 'data'), '--pred-root', str(pred_root),
                '--device', 'cuda', '--precision', 'float16', '--process-res', '504', '--dry-run'],
               cwd=project, env=run_env, check=True)

## 5. Reuse or fetch the pinned DA3 source

The official source revision matches the smoke test. If it is already present, the existing checkout is verified and reused.

In [ ]:
source_root = Path('/content/cw1_da3_smoke_source')
source_url = 'https://github.com/ByteDance-Seed/Depth-Anything-3.git'
source_commit = '3d835ec1a5802d64a8b8b15f817a1ab54809bfe4'
if not (source_root / '.git').is_dir():
    if source_root.exists() and any(source_root.iterdir()):
        raise RuntimeError(f'Unexpected nonempty source folder: {source_root}')
    subprocess.run(['git', 'init', str(source_root)], check=True)
    subprocess.run(['git', '-C', str(source_root), 'remote', 'add', 'origin', source_url], check=True)
present = subprocess.run(['git', '-C', str(source_root), 'cat-file', '-e', source_commit + '^{commit}'],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not present:
    subprocess.run(['git', '-C', str(source_root), 'fetch', '--depth', '1', 'origin', source_commit], check=True)
subprocess.run(['git', '-C', str(source_root), 'checkout', '--detach', source_commit], check=True)
actual = subprocess.check_output(['git', '-C', str(source_root), 'rev-parse', 'HEAD'], text=True).strip()
if actual != source_commit:
    raise RuntimeError('DA3 source revision differs from the pinned version')
(results_root / 'source_commit.json').write_text(json.dumps({'url': source_url, 'commit': source_commit}, indent=2))
print('Pinned DA3 source:', actual)

## 6. Reuse or download the verified metric checkpoint

This checks the same immutable official files used by the smoke test: configuration (847 bytes) and model weights (1,336,734,448 bytes). Existing matching files are reused; interrupted `.partial` transfers can resume. A checksum mismatch stops the run.

In [ ]:
weights_root = Path('/content/cw1_da3_smoke_weights')
weights_root.mkdir(exist_ok=True)
da3_revision = '4010e39f3634a45bc60553321fb49fb760bd594e'
download_specs = [
    ('da3_config', f'https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/{da3_revision}/config.json',
     'da3metric-large/config.json', 847, 'a336f3e76fe375aaae17a9aed9130c9f2aa061535d317ec57dcb2f1f02e1dd53'),
    ('da3_weights', f'https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/{da3_revision}/model.safetensors',
     'da3metric-large/model.safetensors', 1336734448, 'bbea5b0b3ee389849cffa7ddae89de064a90abd2b055fc5aa99aac68db324776'),
]

def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()

def download_checked(url, relative, expected_bytes, expected_sha):
    target = weights_root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file():
        candidate = target
    else:
        candidate = target.with_name(target.name + '.partial')
        if candidate.exists() and candidate.stat().st_size > expected_bytes:
            raise ValueError(f'Oversized partial download: {candidate}; inspect it before retrying')
        if not candidate.is_file() or candidate.stat().st_size != expected_bytes:
            subprocess.run(['curl', '-fL', '--retry', '3', '--continue-at', '-', url, '-o', str(candidate)], check=True)
    if candidate.stat().st_size != expected_bytes:
        raise ValueError(f'Incomplete {relative}; rerun this cell to resume')
    checksum = digest_file(candidate)
    if checksum != expected_sha:
        raise ValueError(f'SHA256 mismatch for {relative}; do not use this file')
    if candidate != target:
        candidate.replace(target)
    return target, checksum

checkpoint_paths, checkpoint_records = {}, []
for key, url, relative, expected_bytes, expected_sha in download_specs:
    target, checksum = download_checked(url, relative, expected_bytes, expected_sha)
    checkpoint_paths[key] = target
    checkpoint_records.append({'key': key, 'url': url, 'file': relative, 'bytes': expected_bytes, 'sha256': checksum})
    (results_root / 'checkpoints.json').write_text(json.dumps(checkpoint_records, indent=2))
    print('Verified', relative, f'({expected_bytes / 1024**2:.1f} MiB)', flush=True)
da3_config = json.loads(checkpoint_paths['da3_config'].read_text())
if da3_config.get('model_name', '').lower() != 'da3metric-large':
    raise ValueError('The DA3 checkpoint is not the required metric model')

## Optional: restore a cache saved from an earlier Colab VM

Skip this cell on the first run. To resume after a VM reset, upload previously downloaded full or partial dataset ZIPs through the Colab **Files** panel, then enter their local paths below. Import verifies their hashes and matches the current RGB/calibration before rebasing association paths. Compatible completed frames are reused; inference still uses the fixed CUDA FP16 configuration for missing frames. Do this before starting the generation cell.

In [ ]:
RESTORE_CACHE_ZIPS = []  # Example: ['/content/da3_kitti06_..._results.zip']
for cache_zip in RESTORE_CACHE_ZIPS:
    subprocess.run([str(python), str(project / 'scripts/part2/transfer_da3_cache.py'), 'import',
                    '--archive', str(Path(cache_zip)), '--pred-root', str(pred_root),
                    '--data-root', str(project / 'data')], cwd=project, env=run_env, check=True)
if RESTORE_CACHE_ZIPS:
    print('Prior cache archives verified and imported; the runner will reuse compatible completed frames.')

## 7. Prepare per-dataset exports and the metadata archive

Each dataset is exported separately with the transfer helper, which verifies the cached NPY/PNG hashes and marks the archive as full or partial. Input images, model weights and source repositories are excluded. The small metadata ZIP contains the environment, hardware, commands, logs and progress.

Large cache ZIPs are announced with their path, size and SHA256. Use Colab's **Files** sidebar, open `/content/da3_full_exports`, right-click a ZIP and choose **Download**. Download them sequentially and confirm each finishes; no large browser download widget starts automatically. The small metadata ZIP can use the download widget. Files remain in the VM until you end it.

In [ ]:
import signal
import time

exports_root = Path('/content/da3_full_exports')
exports_root.mkdir(exist_ok=True)
logs_root = results_root / 'logs'
logs_root.mkdir(exist_ok=True)
progress_path = results_root / 'full_progress.json'
progress = json.loads(progress_path.read_text()) if progress_path.exists() else {}

def stop_child(child):
    if child is None or child.poll() is not None:
        return
    for action, timeout in ((lambda: child.send_signal(signal.SIGINT), 15),
                            (child.terminate, 15), (child.kill, 10)):
        try:
            action()
        except ProcessLookupError:
            return
        try:
            child.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue
    raise RuntimeError('Child did not stop; do not export until its process has ended')

def save_progress():
    temporary = progress_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(progress, indent=2))
    temporary.replace(progress_path)

def download_metadata(path):
    if path.stat().st_size > 50 * 1024**2:
        raise RuntimeError('Metadata archive unexpectedly exceeds 50 MiB; use the Files sidebar to inspect it')
    print('Small metadata ZIP:', path, flush=True)
    try:
        files.download(str(path))
    except Exception as exc:
        print('Metadata download request failed:', exc)
        print('Use the Files sidebar > /content/da3_full_exports > right-click > Download.')

def metadata_archive():
    path = exports_root / 'da3_full_run_metadata.zip'
    with zipfile.ZipFile(path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for item in sorted(results_root.rglob('*')):
            if item.is_file():
                archive.write(item, item.relative_to(results_root.parent))
    return path

def export_sequence_cache(sequence):
    global active_export_process
    export_child = globals().get('active_export_process')
    if isinstance(export_child, subprocess.Popen) and export_child.poll() is None:
        raise RuntimeError('Wait for the previous export child to stop before starting another export')
    child = globals().get('process')
    if isinstance(child, subprocess.Popen) and child.poll() is None:
        raise RuntimeError('Stop and wait for the inference child before exporting a cache snapshot')
    if sequence not in expected_counts:
        raise ValueError(f'Unknown sequence: {sequence}')
    manifest = pred_root / sequence / 'manifest.json'
    if not manifest.is_file():
        print(f'{sequence}: no committed cache manifest yet; metadata can still be downloaded.')
        return None
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    output = exports_root / f'da3_{sequence}_{stamp}_results.zip'
    started = time.perf_counter()
    command = [str(python), str(project / 'scripts/part2/transfer_da3_cache.py'), 'export',
               '--sequence', sequence, '--pred-root', str(pred_root), '--output', str(output)]
    active_export_process = subprocess.Popen(command, cwd=project, env=run_env, stdout=subprocess.PIPE,
                                              stderr=subprocess.STDOUT, text=True, bufsize=1)
    export_result = None
    try:
        for line in active_export_process.stdout:
            print(line, end='', flush=True)
            try:
                candidate = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(candidate, dict) and candidate.get('output') == str(output):
                export_result = candidate
        export_returncode = active_export_process.wait()
    except KeyboardInterrupt:
        stop_child(active_export_process)
        print('Export stopped. The committed depth cache remains; rerun the rescue export when ready.')
        raise
    finally:
        if active_export_process.poll() is not None and active_export_process.stdout is not None:
            active_export_process.stdout.close()
    if export_returncode != 0 or export_result is None:
        print(f'{sequence}: export failed; inspect the error before relying on this archive.')
        return None
    progress.setdefault(sequence, {})['export'] = {
        'output': str(output), 'bytes': output.stat().st_size, 'sha256': export_result['sha256'],
        'export_seconds': time.perf_counter() - started,
        'finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    }
    save_progress()
    print('CACHE ZIP READY:', output, flush=True)
    print(f'Size: {output.stat().st_size / 1024**3:.3f} GiB; SHA256: {export_result["sha256"]}', flush=True)
    print('Files sidebar > /content/da3_full_exports > right-click this ZIP > Download. Wait for completion.', flush=True)
    return output

## 8. Generate KITTI, TUM and Replica, saving each completed dataset

This uses CUDA FP16 throughout, at processing resolution 504. The shorter KITTI sequence runs first so the first export is available sooner. When its path appears, download it manually from the Files sidebar. The runner prints committed frame counts and an ETA about every 25 frames and writes its manifest atomically after every frame. PNG overflow is stored as invalid pixels rather than silently saturating; NPY retains the metric prediction.

Do not disconnect the runtime deliberately while it is generating a sequence. If you stop this cell, the child process is stopped and the last committed cache remains; run the rescue cell below before ending the session. If a sequence fails, its partial cache/logs remain and the other datasets are still attempted with the same settings.

In [ ]:
sequence_order = ['kitti06', 'tum_fr3', 'replica_room0']
for current_sequence in sequence_order:
    command = [str(python), str(project / 'scripts/part2/predict_da3.py'),
               '--sequence', current_sequence, '--checkpoint', str(checkpoint_dir),
               '--source-root', str(source_root), '--data-root', str(project / 'data'),
               '--pred-root', str(pred_root), '--device', 'cuda', '--precision', 'float16',
               '--process-res', '504', '--png-overflow', 'invalid']
    log_path = logs_root / f'{current_sequence}.log'
    previous_export = progress.get(current_sequence, {}).get('export')
    progress[current_sequence] = {'status': 'running', 'expected_frames': expected_counts[current_sequence],
                                  'command': command, 'log': str(log_path),
                                  'started_utc': datetime.datetime.now(datetime.timezone.utc).isoformat()}
    if previous_export is not None:
        progress[current_sequence]['export'] = previous_export
    save_progress()
    print('\nStarting', current_sequence, expected_counts[current_sequence], 'frames', flush=True)
    started = time.perf_counter()
    process = None
    try:
        with log_path.open('a') as log:
            log.write('\nRUN ATTEMPT ' + progress[current_sequence]['started_utc'] + '\n')
            log.flush()
            process = subprocess.Popen(command, cwd=project, env=run_env, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    except KeyboardInterrupt:
        stop_child(process)
        progress[current_sequence].update(status='interrupted', whole_process_seconds=time.perf_counter()-started)
        save_progress()
        metadata_archive()
        print('Stopped. Last committed frames remain. Run the partial-cache rescue cell before ending this VM.')
        raise
    cache_manifest = pred_root / current_sequence / 'manifest.json'
    cache = json.loads(cache_manifest.read_text()) if cache_manifest.exists() else {}
    committed = len(cache.get('frames', {}))
    progress[current_sequence].update(
        status='complete' if returncode == 0 and committed == expected_counts[current_sequence] else 'partial_or_failed',
        returncode=returncode, frames_committed=committed,
        whole_process_seconds=time.perf_counter()-started,
        finished_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
    )
    save_progress()
    print(current_sequence, progress[current_sequence]['status'],
          f'{committed}/{expected_counts[current_sequence]} committed frames', flush=True)
    export_sequence_cache(current_sequence)
    metadata_archive()
    total_committed = sum(row.get('frames_committed', 0) for row in progress.values())
    print('Stored across attempted sequences:', total_committed, '/ 5686', flush=True)
    if returncode != 0:
        print('Failure preserved. The next sequence will use the same CUDA FP16 configuration.')
metadata_archive()
print('All sequences have been attempted. Check completeness below, then download the dataset ZIPs manually one at a time.')

## 9. Check that all 5,686 frames are present and record timings

The status table distinguishes committed cache frames from missing work. Summed stored inference timings exclude RGB reads, saving, cache validation, model startup, ZIP compression and downloads. They may span multiple resumed sessions; preserve the per-session hardware metadata with them. Whole-process wall times are kept separately in the run log metadata.

In [ ]:
import math
import statistics

cache_status = []
for sequence, expected in expected_counts.items():
    path = pred_root / sequence / 'manifest.json'
    manifest = json.loads(path.read_text()) if path.is_file() else {}
    entries = list(manifest.get('frames', {}).values())
    times = [float(row['inference_seconds']) for row in entries
             if row.get('inference_seconds') is not None and math.isfinite(float(row['inference_seconds']))
             and float(row['inference_seconds']) > 0]
    cache_status.append({'sequence': sequence, 'expected_frames': expected,
                         'frames_committed': len(entries), 'complete': len(entries) == expected,
                         'timed_frames': len(times), 'stored_inference_seconds': sum(times),
                         'mean_stored_inference_seconds': statistics.mean(times) if times else None,
                         'manifest': str(path)})
(results_root / 'cache_status.json').write_text(json.dumps(cache_status, indent=2))
print(json.dumps(cache_status, indent=2))
total = sum(row['frames_committed'] for row in cache_status)
if total == 5686 and all(row['complete'] for row in cache_status):
    print('COMPLETE: all 5686 frames have committed DA3 cache entries.')
else:
    print(f'PARTIAL: {total}/5686 committed frames. Resume missing work or export the current partial cache.')
download_metadata(metadata_archive())

## 10. Rescue a partial dataset before terminating a surviving VM

If generation was interrupted, set `EXPORT_PARTIAL_NOW=True` and run this cell to export the last committed frame pairs. It defaults to the most recently attempted sequence; change `RESCUE_SEQUENCE` if needed. Partial archives are labelled by the transfer helper. This cannot recover files after the Colab VM has already been reset or deleted.

In [ ]:
EXPORT_PARTIAL_NOW = False  # Set True after stopping generation to export a partial cache.
RESCUE_SEQUENCE = globals().get('current_sequence', 'kitti06')
if EXPORT_PARTIAL_NOW:
    export_sequence_cache(RESCUE_SEQUENCE)
    download_metadata(metadata_archive())
else:
    print('Rescue export skipped; set EXPORT_PARTIAL_NOW=True only when needed.')

## Bring the caches back to the coursework project

Keep the three dataset ZIPs and the metadata ZIP. The local transfer helper imports each dataset, verifies the cached hashes, checks original RGB identities against the local data, and rebuilds association paths for this computer. Run these from the coursework repository after downloading:

```text
python scripts/part2/transfer_da3_cache.py import --archive <downloaded_kitti_zip> --pred-root depth_maps/da3_metric_large_fp16 --data-root ../data
python scripts/part2/transfer_da3_cache.py import --archive <downloaded_tum_zip> --pred-root depth_maps/da3_metric_large_fp16 --data-root ../data
python scripts/part2/transfer_da3_cache.py import --archive <downloaded_replica_zip> --pred-root depth_maps/da3_metric_large_fp16 --data-root ../data
```

Return the downloaded files before choosing later SLAM settings. Full depth accuracy and temporal consistency can then be evaluated locally with the available GT.

**Validation:** notebook schema, code and embedded inventory logic are checked locally. The full GPU run, Colab upload, installation, memory fit, inference and browser downloads have not been executed here; they require your Colab session.